In [ ]:
from pathlib import Path
import os
import zipfile
import urllib.request

GITHUB_USER = "Facuriy"
GITHUB_REPO = "agro-ai-kurs"
BRANCH = "main"
BLOCK_DIR = Path("01_fernerkundung_drohne_indices_DE")
ZIP_NAME = "01_fernerkundung_drohne_indices_DE.zip"
ZIP_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}/raw/{BRANCH}/{ZIP_NAME}"
RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{BRANCH}/01_fernerkundung_drohne_indices_DE"
DATA_FILE = Path("class_data_remote_sensing/remote_sensing_small.npz")
PREVIEW_FILE = Path("class_data_remote_sensing/preview_rgb.jpg")

def in_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False

def find_course_folder():
    candidates = [Path("."), BLOCK_DIR]
    candidates += [p.parent.parent for p in Path(".").rglob("remote_sensing_small.npz")]
    for candidate in candidates:
        if (candidate / DATA_FILE).exists():
            return candidate
    return None

def download_direct_files():
    Path("class_data_remote_sensing").mkdir(parents=True, exist_ok=True)
    files = [DATA_FILE, PREVIEW_FILE]
    for rel_path in files:
        target = Path(rel_path)
        if not target.exists():
            url = f"{RAW_BASE}/{rel_path.as_posix()}"
            print("Lade Datei:", url)
            urllib.request.urlretrieve(url, target)

course_folder = find_course_folder()

if course_folder is None and in_colab():
    print("Lade Unterrichtsdaten:", ZIP_URL)
    urllib.request.urlretrieve(ZIP_URL, ZIP_NAME)
    try:
        with zipfile.ZipFile(ZIP_NAME, "r") as zf:
            zf.extractall(".")
            names = zf.namelist()
            print("ZIP entpackt:", len(names), "Dateien")
    except zipfile.BadZipFile:
        print("ZIP konnte nicht gelesen werden. Direkter Download wird benutzt.")
    course_folder = find_course_folder()

if course_folder is None and in_colab():
    download_direct_files()
    course_folder = find_course_folder()

if course_folder is None:
    found = [str(p) for p in Path(".").rglob("*")][:30]
    raise FileNotFoundError("Die Unterrichtsdaten wurden nicht gefunden. Gefundene Dateien: " + str(found))

os.chdir(course_folder)

print("Arbeitsordner:", Path.cwd())
print("Datensatz:", DATA_FILE)


# Fernerkundung, Drohne und Vegetationsindizes

Dieses Notebook ist für den Kurs vorbereitet. Es nutzt kleine, schnelle Unterrichtsdaten.

## Hinweise zur Übung

**Ziel der Einheit:** Die Studierenden sollen verstehen, dass ein Fernerkundungsbild kein normales Foto ist, sondern ein Datenwürfel aus Pixeln und Spektralbändern.

**Empfohlener Ablauf (60-90 min):**

1. RGB und Falschfarbe vergleichen.
2. NDVI und NDRE gemeinsam herleiten.
3. Schwellenwerte in der `AUSPROBIEREN`-Zelle variieren lassen.
4. Zonenbericht diskutieren: Welche Zone würde man zuerst im Feld prüfen?

**Erwartete Kernbotschaften:**

- NDVI/NDRE sind Hinweise, keine Diagnose.
- Drohne = Detail; Satellit = Fläche und Wiederholung.
- Geringere Auflösung mischt Pflanze, Boden und Schatten in einem Pixel.
- Ein niedriger Index kann Krankheit, Trockenstress, Schatten oder Boden bedeuten.

## Erwartete Beobachtungen und Diskussion

**Typische Beobachtungen:**

- Falschfarbe hebt Vegetation stärker hervor als RGB.
- NDVI zeigt Vegetation deutlich, verliert aber Detail bei simulierter grober Auflösung.
- Der Alarmkarten-Schwellenwert verändert die Interpretation stark.

**Diskussionsfragen:**

- Ist eine rote Alarmzone automatisch krank?
- Welche Zusatzdaten bräuchte man für eine biologische Diagnose?
- Wann ist Satellit besser als Drohne, obwohl die Auflösung schlechter ist?

# Fernerkundung mit Drohne und Satellit: Vegetationsindizes verstehen

### Mini-Praxisklasse für Studierende der Agrarwissenschaften

In dieser Klasse arbeiten wir mit einem **sehr kleinen, komprimierten Multispektral-Ausschnitt**.
Das Ziel ist nicht perfekte Bildqualität, sondern schnelles Lernen:

- Was ist ein Rasterbild?
- Was sind Spektralbänder?
- Was bedeuten RGB, RedEdge und NIR?
- Wie berechnet man NDVI und NDRE?
- Warum sehen Drohne und Satellit unterschiedlich aus?
- Wie kann man mit Schwellenwerten einfache Stresskarten erzeugen?

Das Notebook ist Colab-freundlich: keine großen GeoTIFFs, kein `rasterio` nötig, nur ein kleines `.npz`.

## 1. Begriffe: Was ist Fernerkundung?

| Begriff | Einfache Erklärung |
|---|---|
| **Fernerkundung** | Pflanzen/Flächen messen, ohne sie direkt zu berühren, z. B. mit Drohne oder Satellit. |
| **Raster** | Ein Bild als Gitter aus Pixeln. Jedes Pixel speichert Messwerte. |
| **Band / Kanal** | Eine Messung in einem Wellenlängenbereich, z. B. Rot oder NIR. |
| **RGB** | Normales Farbbild: Rot, Grün, Blau. |
| **NIR** | Nahinfrarot. Gesunde Pflanzen reflektieren hier oft stark. |
| **RedEdge** | Übergangsbereich zwischen Rot und NIR, oft sensibel für Pflanzenstress. |
| **Vegetationsindex** | Rechenregel aus Bändern, um Pflanzenzustand sichtbar zu machen. |

Eine Drohne sieht oft viele Details. Ein Satellit sieht größere Flächen, aber gröber.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("class_data_remote_sensing/remote_sensing_small.npz")
if not DATA.exists():
    matches = list(Path(".").rglob("remote_sensing_small.npz"))
    if not matches:
        raise FileNotFoundError("remote_sensing_small.npz wurde nicht gefunden. Bitte oben Runtime > Disconnect and delete runtime wählen und das Notebook neu öffnen.")
    DATA = matches[0]
    print("Daten gefunden:", DATA)

OUT = DATA.parents[1] / "class_outputs_remote_sensing"
OUT.mkdir(exist_ok=True)

data = np.load(DATA, allow_pickle=True)
cube = data["cube"].astype("float32")       # Reihenfolge: blue, green, red, rededge, nir
band_names = [str(name) for name in data["band_names"]]
bands = {name: cube[i] for i, name in enumerate(band_names)}
if "zones" in data.files:
    zones = data["zones"]
else:
    h, w = cube.shape[1], cube.shape[2]
    zones = np.zeros((h, w), dtype="int32")
    for row in range(3):
        for col in range(3):
            y0, y1 = int(row*h/3), int((row+1)*h/3)
            x0, x1 = int(col*w/3), int((col+1)*w/3)
            zones[y0:y1, x0:x1] = row*3 + col

print("Cube:", cube.shape)
print("Bänder:", band_names)
print("Arbeitsordner:", Path.cwd())


## 2. Das Bild sichtbar machen

Multispektraldaten sind nicht automatisch ein Foto. Für ein **RGB-Bild** wählen wir:

```text
Rot   = red
Grün  = green
Blau  = blue
```

Für ein **Falschfarbenbild** wählen wir oft:

```text
Rot   = nir
Grün  = red
Blau  = green
```

Dann leuchtet Vegetation häufig rötlich, weil gesunde Blätter im NIR stark reflektieren.

In [ ]:
def norm01(x, p1=2, p2=98):
    lo, hi = np.nanpercentile(x, [p1, p2])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

def rgb_stack(r, g, b):
    return np.dstack([norm01(r), norm01(g), norm01(b)])

true_rgb = rgb_stack(bands["red"], bands["green"], bands["blue"])
false_color = rgb_stack(bands["nir"], bands["red"], bands["green"])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(true_rgb); ax[0].set_title("RGB: red/green/blue")
ax[1].imshow(false_color); ax[1].set_title("Falschfarbe: NIR/red/green")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 3. Vegetationsindizes: NDVI und NDRE

### NDVI

$$NDVI = \frac{NIR - Rot}{NIR + Rot}$$

Gesunde Pflanzen: NIR hoch, Rot niedrig → NDVI hoch.

### NDRE

$$NDRE = \frac{NIR - RedEdge}{NIR + RedEdge}$$

NDRE nutzt RedEdge und kann bei dichter Vegetation oder Stress manchmal informativer sein.

**Achtung:** Ein Index ist ein Hinweis, keine Diagnose. Niedriger NDVI kann Krankheit, Trockenheit, Schatten, Boden oder Mischpixel bedeuten.

In [ ]:
def safe_index(a, b):
    return (a - b) / (a + b + 1e-6)

ndvi = safe_index(bands["nir"], bands["red"])
ndre = safe_index(bands["nir"], bands["rededge"])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
im0 = ax[0].imshow(ndvi, cmap="RdYlGn", vmin=-0.1, vmax=0.9)
ax[0].set_title("NDVI")
plt.colorbar(im0, ax=ax[0], fraction=0.046)
im1 = ax[1].imshow(ndre, cmap="viridis", vmin=-0.1, vmax=0.6)
ax[1].set_title("NDRE")
plt.colorbar(im1, ax=ax[1], fraction=0.046)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

print("NDVI Mittelwert:", round(float(np.nanmean(ndvi)), 3))
print("NDRE Mittelwert:", round(float(np.nanmean(ndre)), 3))

## 4. Drohne vs. Satellit: Auflösung simulieren

Wir haben hier nur einen kleinen Ausschnitt. Trotzdem können wir die Idee zeigen:

- **Drohne:** kleine Pixel, viele Details.
- **Satellit:** größere Pixel, weniger Details, aber große Fläche.

Wir simulieren grobe Satellitenauflösung, indem wir Pixelblöcke mitteln.

In [ ]:
def block_mean(img, factor):
    h, w = img.shape
    h2, w2 = (h // factor) * factor, (w // factor) * factor
    x = img[:h2, :w2]
    return x.reshape(h2//factor, factor, w2//factor, factor).mean(axis=(1, 3))

def nearest_up(img, factor):
    return np.repeat(np.repeat(img, factor, axis=0), factor, axis=1)

ndvi_drone = ndvi
ndvi_sat10 = nearest_up(block_mean(ndvi, 4), 4)
ndvi_sat30 = nearest_up(block_mean(ndvi, 12), 12)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, img, title in zip(ax, [ndvi_drone, ndvi_sat10, ndvi_sat30],
                         ["Drohne: fein", "Satellit: mittel", "Landsat-artig: grob"]):
    im = a.imshow(img, cmap="RdYlGn", vmin=-0.1, vmax=0.9)
    a.set_title(title); a.axis("off")
plt.tight_layout(); plt.show()

## 5. AUSPROBIEREN: einfache Stresskarte

Jetzt machen wir eine einfache Regel:

```text
Alarm = NDVI kleiner als Schwellenwert
```

Das ist keine Krankheitsdiagnose. Es ist eine **Priorisierungskarte**: Wo würde ich zuerst genauer hinschauen?

In [ ]:
# AUSPROBIEREN
NDVI_SCHWELLE = 0.50
NDRE_SCHWELLE = 0.22

vegetation = ndvi > 0.20
alarm = vegetation & ((ndvi < NDVI_SCHWELLE) | (ndre < NDRE_SCHWELLE))
alarm_pct = 100 * alarm.sum() / max(1, vegetation.sum())
print(f"Alarmfläche innerhalb Vegetation: {alarm_pct:.1f}%")

overlay = true_rgb.copy()
overlay[alarm] = [1.0, 0.05, 0.05]

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(true_rgb); ax[0].set_title("RGB")
ax[1].imshow(alarm, cmap="Reds"); ax[1].set_title("Alarmmaske")
ax[2].imshow(overlay); ax[2].set_title("Overlay")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 6. Metriken pro Zone

Wir teilen das Bild in Zonen und berechnen:

- Anzahl Vegetationspixel
- Anzahl Alarmpixel
- Prozent Alarm
- Empfehlung für Feldbegehung

Das ist ein typischer Schritt von Bildanalyse zu agronomischer Entscheidung.

In [ ]:
rows = []
n_rows, n_cols = 3, 3
h, w = ndvi.shape
for i in range(n_rows):
    for j in range(n_cols):
        y0, y1 = int(i*h/n_rows), int((i+1)*h/n_rows)
        x0, x1 = int(j*w/n_cols), int((j+1)*w/n_cols)
        veg = vegetation[y0:y1, x0:x1]
        al = alarm[y0:y1, x0:x1]
        pct = 100 * (al & veg).sum() / max(1, veg.sum())
        rows.append({
            "Zone": f"{chr(65+i)}{j+1}",
            "Vegetationspixel": int(veg.sum()),
            "Alarmpixel": int((al & veg).sum()),
            "Alarm_%": round(float(pct), 1),
            "Empfehlung": "zuerst prüfen" if pct > 30 else ("beobachten" if pct > 15 else "niedrig"),
        })

report = pd.DataFrame(rows).sort_values("Alarm_%", ascending=False)
display(report)
report.to_csv(OUT / "zonen_report.csv", index=False)
print("Gespeichert:", OUT / "zonen_report.csv")

## 7. Grenzen und Diskussion

- NDVI/NDRE zeigen Pflanzenzustand, aber nicht automatisch die Ursache.
- Schatten, Boden, Mischpixel und Beleuchtung können Indizes verändern.
- Drohnen sind detailreich, aber lokal. Satelliten sind grober, aber regelmäßig und großflächig.
- Kleine Daten sind gut zum Lernen, aber nicht genug für Forschung.

**Kernbotschaft:** Fernerkundung ist eine Messung. Die biologische Interpretation braucht Agronomie.